# DeepD Human HBB best-layer OVR demo

This notebook reproduces one-vs-rest predictions across the complete `chr11:5,220,000–5,240,000` HBB interval. It runs 12 real models on eight real `(20000, 576)` MoE feature arrays and invokes the original publication-style track renderer.


## 1. Locate or download runtime assets

Local `data/` and `models/` directories are used when present. If either is incomplete, `ensure_demo_assets` downloads both subtrees from the Hugging Face dataset `biomap-research/DeepD` (`interpretability/` prefix). Override with `DEEPD_DATA_REPO_ID` or function arguments only if you host a fork.


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display

from hbb_ovr_demo import ensure_demo_assets, run_hbb_demo

cwd = Path.cwd().resolve()
ROOT = next(
    candidate for candidate in (
        cwd,
        cwd / 'interpretability',
        cwd.parent,
    )
    if (candidate / 'hbb_ovr_demo.py').is_file()
)
asset_status = ensure_demo_assets(ROOT)
print(f'Demo root: {ROOT}')
print(f"Assets downloaded in this run: {asset_status['downloaded']}")


## 2. Run live inference, verification, and the original plotter

The consolidated module validates model hashes and feature shapes, recomputes all 240,000 model-row probabilities, performs numerical regression against the original predictions, and creates separate 8 kb and 1M figures. XGBoost 3.2.0 is used when available; otherwise the verified NumPy predictor is used.


In [ ]:
result = run_hbb_demo(ROOT, download_if_missing=False)
report = result['report']
display(pd.DataFrame([report]))
display(result['prediction_summary'].round(6))


## 3. Original-style HBB tracks

The original command plots five labels and excludes `cCRE`; the `cCRE` models are still included in live inference and numerical verification.


In [ ]:
display(SVG(filename=str(result['plot_paths']['8kb_svg'])))


In [ ]:
display(SVG(filename=str(result['plot_paths']['1m_svg'])))


## 4. Full-test best-layer performance

HBB interval metrics are local diagnostics. Primary model performance should be reported from the complete held-out test set.


In [ ]:
display(SVG(filename=str(result['best_chart'])))


## 5. Outputs and reproduction scope

All newly generated predictions, summaries, reports, staged plot inputs, SVG files, and PDF files are written under `generated/`. Large feature arrays and model JSONs are fetched from `biomap-research/DeepD` when they are not already present locally.
